# The stories

This notebook goes with `vak`, step 2. It starts with five stories typed into a cell, small enough to check by
hand. Then it runs the same functions on the real file that `uv run vak data` reads.

Before this: nothing. Every later step starts from these stories. After this: step 3 turns each story into tokens.

Sections 2 to 7 read the downloaded file, 249 MB. Run `uv run vak data` once first. Otherwise `download()` in
section 2 fetches the file itself.

The map: each row is one step that `load_split()` takes, in order.

| Step | What happens | Function | Section |
|---|---|---|---|
| file | one file of TinyStories, downloaded once into `data/` | `download()` | 2 |
| stories | one story per row, in one text column | `pyarrow.parquet` | 2, 3, 4 |
| drop repeats | keep each story once, and drop blank ones | `unique_stories()` | 1, 5 |
| split by a seed | pick the stories to train on and the stories to hold out | `split()` | 1, 5 |
| train / held out | 50,000 and 2,000 stories, the same for every command | `load_split()` | 6, 7 |

## 1. Five stories, small enough to check by hand

A **story** is one short text: one row of the file. Before any model sees the stories, `unique_stories` keeps each
story once and drops blank ones. Here are five stories typed by hand. The third repeats the first, and the fourth
is blank.

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # a note about progress bars; nothing here needs them
from vak.data import unique_stories

toy_stories = [
    "Tom has a red ball.",
    "Mia sees a big dog.",
    "Tom has a red ball.",  # the first story again
    "   ",  # blank: only spaces
    "The sun is hot.",
]
toy_unique = unique_stories(toy_stories)
print(len(toy_stories), "stories in,", len(toy_unique), "out:")
for story in toy_unique:
    print(" ", story)

5 stories in, 3 out:
  Tom has a red ball.
  Mia sees a big dog.
  The sun is hot.


**What you see**

`5 stories in, 3 out`. The repeat and the blank story are gone. The three left keep their order from the list:
`Tom has a red ball.` first and `The sun is hot.` last.

Next, `split` picks the stories to train on and the stories to **hold out**. No model learns from a held-out
story: it is kept for the test. The picks are random, and a **seed** fixes them. The same seed gives the same
split every time. Here the split runs three times: twice with seed 0, then once with seed 1.

In [2]:
from vak.data import split

for seed in [0, 0, 1]:
    toy_parts = split(toy_unique, n_train=2, n_held_out=1, seed=seed)
    print(f"seed {seed}: train {toy_parts.train}, held out {toy_parts.held_out}")

seed 0: train ['Mia sees a big dog.', 'The sun is hot.'], held out ['Tom has a red ball.']
seed 0: train ['Mia sees a big dog.', 'The sun is hot.'], held out ['Tom has a red ball.']
seed 1: train ['Tom has a red ball.', 'The sun is hot.'], held out ['Mia sees a big dog.']


**What you see**

Both runs with seed 0 give the same split: they hold out `['Tom has a red ball.']`. Seed 1 picks again and holds
out `['Mia sees a big dog.']`. No story is in both parts, because the split picks each story at most once.

Used in: `uv run vak data` runs the same two functions on the real file, with seed 0.

## 2. The real file

The stories come from TinyStories: short stories in simple English, written by GPT-3.5 and GPT-4. `download()`
fetches the first of its four training files into `data/`, once. Later calls find it there. The file is in
**parquet** format: a table stored column by column. Its metadata gives the number of rows before we read any
story.

In [3]:
import pyarrow.parquet as pq
from vak.data import download

path = download()  # downloads the file on the first call only
print("file:", path.name)
print(f"size: {path.stat().st_size:,} bytes, about {round(path.stat().st_size / 1e6)} MB")
print(f"rows: {pq.read_metadata(path).num_rows:,}")
print("columns:", pq.read_schema(path).names)

file: train-00000-of-00004-2d5a1467fff1081b.parquet
size: 248,731,111 bytes, about 249 MB
rows: 529,930
columns: ['text']


**What you see**

The file holds `248,731,111 bytes`, `about 249 MB`, in `529,930` rows. It has one column, `['text']`. Each row is
one story, as plain text.

Now read the text column into a Python list, the same way `load_split` does. Then print the first story whole.

In [4]:
stories = pq.read_table(path, columns=["text"]).column("text").to_pylist()
print(f"{len(stories):,} stories in the list")
print()
print(stories[0])

529,930 stories in the list

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


**What you see**

`529,930 stories`, one per row. The first one is about Lily and a `needle`. It has three paragraphs with a blank
line between them, and the people in it talk in quotes.

Used in: `load_split()` reads the file the same way, for every command.

## 3. One story close up

We can count a story's length in characters or in words. Here a **word** is a piece of text between spaces or
new lines, and `str.split()` cuts the story there. A word keeps its punctuation.

In [5]:
story = stories[0]
words = story.split()  # cut at every space and new line
print(len(story), "characters")
print(len(words), "words")
print("the first 10 words:", words[:10])
print("the last 3 words:", words[-3:])

701 characters
134 words
the first 10 words: ['One', 'day,', 'a', 'little', 'girl', 'named', 'Lily', 'found', 'a', 'needle']
the last 3 words: ['and', 'worked', 'together.']


**What you see**

The first story has `701 characters` and `134 words`. A word keeps its punctuation, so `'day,'` is one word, and
so is `'together.'`.

Used in: step 3 measures the same stories in tokens. A token is often a word, or a piece of one.

## 4. Story lengths

One story is not the whole file. This cell counts the words of every story, then sorts the counts. The **median**
is the count in the middle of the sorted list: half the stories are shorter, and half are longer.

In [6]:
lengths = []
for story in stories:
    lengths.append(len(story.split()))
lengths.sort()
print(f"{len(lengths):,} stories")
print("shortest:", lengths[0], "words")
print("median:  ", lengths[len(lengths) // 2], "words")  # the middle of the sorted list
print("longest: ", lengths[-1], "words")
print("stories with 0 words:", lengths.count(0))

529,930 stories
shortest: 0 words
median:   153 words
longest:  963 words
stories with 0 words: 55


**What you see**

The median story has `153 words`, and the longest has `963 words`. The shortest has `0 words`: `55` stories are
blank. Section 5 finds them again.

A text **histogram** shows how the lengths spread. Each row counts the stories in a range of 50 words. Each `#`
stands for 5,000 stories.

In [7]:
rows = [0] * 11  # row 0: 0 to 49 words, row 1: 50 to 99 words, and so on; row 10: 500 words or more
for length in lengths:
    row = length // 50
    if row > 10:
        row = 10
    rows[row] += 1

for row in range(11):
    if row < 10:
        label = f"{row * 50} to {row * 50 + 49}"
    else:
        label = "500 or more"
    bar = "#" * (rows[row] // 5_000)
    print(f"{label:>11} words {rows[row]:>8,}  {bar}")

    0 to 49 words       96  
   50 to 99 words   18,631  ###
 100 to 149 words  226,491  #############################################
 150 to 199 words  177,917  ###################################
 200 to 249 words   41,398  ########
 250 to 299 words   21,483  ####
 300 to 349 words   16,830  ###
 350 to 399 words   11,203  ##
 400 to 449 words    6,805  #
 450 to 499 words    4,024  
500 or more words    5,052  #


**What you see**

Most stories have `100 to 149` words: `226,491` of them. The next row, `150 to 199`, holds `177,917`. Only `96`
stories have fewer than 50 words. The long stories thin out slowly: `5,052` have 500 words or more.

Used in: step 3 counts the same stories in tokens.

## 5. Repeats, and the leak they cause

Some stories are in the file more than once, and some are blank. This cell counts both. `Counter` counts how often each
text appears. Like `unique_stories`, it strips the spaces at each end of a story first.

In [8]:
from collections import Counter

blank = 0
copies = Counter()  # each story's text -> how many times the file holds it
for story in stories:
    text = story.strip()
    if text:
        copies[text] += 1
    else:
        blank += 1

repeated = []
for text, count in copies.items():
    if count > 1:
        repeated.append(text)
print(f"{len(stories):,} stories in the file")
print(f"{blank:,} are blank")
print(f"{len(repeated):,} stories are in the file more than once")
print("the most copies of one story:", max(copies.values()))

529,930 stories in the file
55 are blank
20,250 stories are in the file more than once
the most copies of one story: 2


**What you see**

`55 are blank`, and `20,250 stories` are in the file more than once. No story has more than `2` copies. So the
file holds 20,250 extra copies.

Here is the first repeated story, with the rows where it appears.

In [9]:
example = repeated[0]
found_at = []
for row, story in enumerate(stories):
    if story.strip() == example:
        found_at.append(row)
print(f"{copies[example]} copies, at rows {found_at}")
print()
print(example)

2 copies, at rows [900, 438369]

Anna and Ben are twins. They like to play with their toys. They have many toys, but they have one favorite toy. It is a big gray elephant. The elephant is soft and has a long nose. The twins call him Elly.

One day, Anna and Ben want to play with Elly at the same time. They both grab his nose and pull. Elly makes a loud noise. He does not like it. Anna and Ben struggle. They do not want to share.

"Elly is mine!" Anna says.

"No, Elly is mine!" Ben says.

They pull harder. Elly makes a louder noise. He does not like it more.

Mom hears the noise. She comes to the room. She sees Anna and Ben struggle. She sees Elly make noise. She is not happy.

"Anna and Ben, stop!" Mom says. "You are hurting Elly. He is not happy. You have to share. Elly is for both of you."

Anna and Ben look at Mom. They look at Elly. They see his sad eyes. They feel sorry.

"Sorry, Elly," Anna says.

"Sorry, Elly," Ben says.

They let go of his nose. They hug him. Elly makes a happy

**What you see**

The first repeated story is in the file as `2 copies`, at rows `[900, 438369]`. Rows count from 0, as in a Python
list. Both copies are the same text, word for word: a story about two twins and their toy elephant.

`unique_stories` drops the blank stories and the extra copies. 55 + 20,250 = 20,305 stories should go.

In [10]:
unique = unique_stories(stories)
print(f"{len(stories):,} stories in, {len(unique):,} out: {len(stories) - len(unique):,} dropped")

529,930 stories in, 509,625 out: 20,305 dropped

**What you see**

`20,305 dropped`, as counted above. `509,625` stories are left: the same number that the command logs.

Why drop repeats? A story that is in the file twice could land in both parts of the split. The test would then
score a story the model trained on. That is a **leak**. This cell splits the stories both ways, with seed 0. Then it
counts the held-out stories that are also training stories.

In [11]:
from vak.data import HELD_OUT_STORIES, SEED, TRAIN_STORIES

for label, pool in [("without dropping repeats", stories), ("after unique_stories", unique)]:
    trial = split(pool, TRAIN_STORIES, HELD_OUT_STORIES, SEED)
    train = set(trial.train)  # a set finds a story fast
    in_both = 0
    for story in trial.held_out:
        if story in train:
            in_both += 1
    print(f"{label}: {in_both} of {len(trial.held_out):,} held-out stories are also training stories")

without dropping repeats: 12 of 2,000 held-out stories are also training stories
after unique_stories: 0 of 2,000 held-out stories are also training stories


**What you see**

Without dropping repeats, `12 of 2,000` held-out stories are also training stories: the test would score 12
stories that the model trained on. After `unique_stories`, `0 of 2,000` are.

Used in: `load_split()` runs `unique_stories` before it splits, for every command.

## 6. The real split

`load_split()` takes all the steps above in one call: download once, read, drop repeats, then split with seed 0.
Every command calls it. So every model learns from the same 50,000 stories, and the test uses the same 2,000.

In [12]:
from vak.data import load_split

parts = load_split()
in_both = set(parts.train) & set(parts.held_out)
print(f"{len(parts.train):,} stories to train on, {len(parts.held_out):,} held out, {len(in_both)} in both")
print()
print(f"held-out story 0, {len(parts.held_out[0].split())} words:")
print(parts.held_out[0])

50,000 stories to train on, 2,000 held out, 0 in both

held-out story 0, 250 words:
Lily and Max were playing in the park. They saw a big lady with a big purse. The purse had many colors and shiny things. Lily liked the purse. She wanted to touch it.

"Can I see your purse, please?" Lily asked the lady.

The lady did not speak. She looked angry. She walked away fast.

Max was scared. He did not like the lady. He thought she was stupid.

"Let's go, Lily. She is not nice," Max said.

But Lily did not listen. She ran after the lady. She wanted the purse.

The lady saw Lily. She was very angry. She yelled at Lily.

"Go away, you little brat! This is my purse! Don't touch it!"

She pushed Lily. Lily fell down. She hurt her knee. She cried.

Max ran to Lily. He hugged her. He helped her stand up.

"I'm sorry, Lily. Are you okay?" Max asked.

Lily nodded. She was sad. She did not understand why the lady was so mean.

"Let's go home, Max. I don't like the park anymore," Lily said.

Max agreed.

**What you see**

`50,000 stories to train on, 2,000 held out, 0 in both`: the same line that the command prints. The first held-out
story has `250 words`. No model learns from it. The test scores every model on it.

Used in: `vak tokens` learns its pieces from the 50,000 training stories. `vak counts`, and later the GPT, report
their loss on the 2,000 held-out stories.

## 7. Explore

The two cells below are made for you to edit. Change `i` to read another training story, or `word` to search for
another word. Then run the cell again. The saved outputs show the values as they are now.

In [13]:
i = 0  # change me: any number from 0 to 49,999
story = parts.train[i]
print(f"training story {i}: {len(story):,} characters, {len(story.split())} words")
print()
print(story)

training story 0: 638 characters, 133 words

Once upon a time, there was a bird. The bird was very happy because it could fly high in the sky. One day, the bird saw a fancy yard with lots of pretty flowers. The bird wanted to see the yard up close, so it flew down to the ground. 

As the bird landed, it saw a little girl playing in the yard. The girl was very happy to see the bird and she smiled. The bird hopped around the yard and looked at all the pretty flowers. 

After a while, the bird flew back up into the sky. It was happy to have seen the fancy yard and the little girl. From that day on, the bird would fly over the yard every day just to say hello to the little girl.


**What you see**

Training story `0` has `638 characters` and `133 words`. The command prints the same story as "Training story 1",
because it counts from 1. Set i to another number and run the cell again.

In [14]:
word = "dragon"  # change me: any word, or any piece of text
found = []
for story in parts.train:
    if word in story:
        found.append(story)
print(f'{len(found):,} of {len(parts.train):,} training stories contain "{word}"')
for story in found[:2]:
    print()
    print(story)

406 of 50,000 training stories contain "dragon"

Once upon a time there was a dragon. He liked flying around and eating lots of treats.
One day, the dragon was feeling sad. He didn't understand why he was so hurt. He thought maybe he was just hungry.
So the dragon went looking for a snack. He flew across the sky in search of food. He found some delicious berries and gobbled them up.

But the dragon still felt hurt and unhappy. He flew around the world until he found a friend. His friend was a wise owl. The owl listened to the dragon and told him what he needed to understand.

The dragon realized that being hurt sometimes is just a part of life. He said goodbye to the owl and thanked him for making him feel better.

As the dragon flew away, he felt relieved and happy. He understood that feeling hurt was ok, but it was also important to find someone who could help.

Once upon a time, there was a little girl named Lily. She loved to draw and always used a big crayon to make a huge line on

**What you see**

`406 of 50,000` training stories contain "dragon". The cell looks for the text anywhere in a story, so "dragons"
counts too. Capitals matter: "Dragon" does not match "dragon". Try a name, such as "Lily", or a word you expect to
be rare.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Drop repeats | `unique_stories` keeps each story once, in file order, and drops blank ones | `load_split()` runs it before the split |
| Held out | stories that no model learns from: the test | `vak counts` and the GPT report their loss on the 2,000 |
| The seed | the same seed gives the same split, every time | every command uses seed 0 |
| The file | one parquet file of TinyStories: 529,930 stories in one text column | `load_split()` reads it the same way |
| Lengths | median 153 words, from 0 (blank) to 963 | step 3 measures the same stories in tokens |
| The leak | 20,250 stories are in the file twice; without dropping repeats, 12 held-out stories are also training stories | the reason the split starts from unique stories |
| The split | `load_split()`: 50,000 to train on, 2,000 held out, 0 in both | every later command starts from it |